# Model Evaluation and Business Analysis

## General Description

This notebook focuses on evaluating the performance of the trained rice leaf disease classification model and analyzing how effectively the results satisfy the project business requirements.

The notebook includes quantitative model evaluation, visual prediction analysis, misclassification investigation, and interpretation of model behavior from both technical and business perspectives. The evaluation process aims to identify the strengths and limitations of the baseline model and assess its practical applicability for rice disease detection tasks.

The outputs generated in this notebook will support the final dashboard, project conclusions, and stakeholder communication.


## Objectives

- Load and evaluate the trained image classification model.
- Generate predictions on validation data.
- Measure model performance using classification metrics.
- Create confusion matrix visualizations.
- Analyze correctly and incorrectly classified examples.
- Investigate potential causes of model errors.
- Evaluate how well the model satisfies business requirements.
- Identify possible improvements for future model development.


## Inputs

Generated model artifacts and processed datasets:

- `outputs/models/rice_leaf_disease_classifier.keras`
- `outputs/models/class_indices.json`
- `outputs/models/training_history.csv`

Processed datasets:
- `inputs/datasets/processed/train_labels.csv`
- `inputs/datasets/processed/val_labels.csv`

Raw image dataset:
- `inputs/datasets/raw/rice/`


## Outputs

Evaluation artifacts and business analysis outputs saved to:

`outputs/evaluation/`

Including:
- confusion matrix visualizations
- classification report
- prediction analysis outputs
- misclassification examples
- evaluation summaries


## Additional Comments

- Dataset imbalance may influence prediction quality for minority disease classes.
- Some disease categories may share similar visual characteristics, increasing classification difficulty.
- Evaluation focuses not only on technical accuracy but also on practical usefulness for agricultural disease detection workflows.


### 1. Load Python Packages and Project Dependencies

This section imports the libraries and dependencies required for model evaluation, prediction analysis, visualization, and business interpretation.

In [1]:
# Data handling
import pandas as pd
import numpy as np

# File and path handling
from pathlib import Path
import os
import json
import random

# Visualisation
import matplotlib.pyplot as plt

# Image processing
from PIL import Image

# TensorFlow / Keras
import tensorflow as tf

from tensorflow.keras.models import load_model

from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator
)

# Evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# Confusion matrix visualization
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


### 2. Set Working Directory

Configure the project root directory and define reusable dataset, model, and output paths used throughout the evaluation workflow.

In [2]:
def setup_root():
    current_path = Path().resolve()
    # Search upwards for project root
    while not (current_path / "requirements.txt").exists():
        if current_path == current_path.parent:
            raise FileNotFoundError(
                "Project root not found."
            )
        current_path = current_path.parent
    return current_path


# Set project root
PROJECT_ROOT = setup_root()

# Change working directory
os.chdir(PROJECT_ROOT)
print(f"Project root set to: {PROJECT_ROOT}")


# Define paths
PROCESSED_DATA_DIR = Path(
    "inputs/datasets/processed"
)

MODEL_OUTPUT_DIR = Path(
    "outputs/models"
)

EVALUATION_OUTPUT_DIR = Path(
    "outputs/evaluation"
)

# Create output directory
EVALUATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Processed dataset directory: "
      f"{PROCESSED_DATA_DIR}")

print(f"Model output directory: "
      f"{MODEL_OUTPUT_DIR}")

print(f"Evaluation output directory: "
      f"{EVALUATION_OUTPUT_DIR}")

Project root set to: C:\code\ml\workspace\rice_leaf_diseases_analyser
Processed dataset directory: inputs\datasets\processed
Model output directory: outputs\models
Evaluation output directory: outputs\evaluation


### 3. Load Trained Model and Datasets

Load the trained classification model, processed validation dataset, and saved class label mappings required for evaluation and prediction analysis.

In [3]:
# Define model paths
model_path = (
    MODEL_OUTPUT_DIR / "rice_leaf_disease_classifier.keras"
)

mapping_path = (
    MODEL_OUTPUT_DIR / "class_indices.json"
)

validation_dataset_path = (
    PROCESSED_DATA_DIR / "val_labels.csv"
)

# Load trained model
model = load_model(model_path)

print("Model loaded successfully.")

Model loaded successfully.


In [4]:
# Load validation dataset
df_val = pd.read_csv(validation_dataset_path)

print(f"Validation samples: {len(df_val)}")

display(df_val.head())

Validation samples: 1730


,image_path,class_id,class_name
0,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut
1,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut
2,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut
3,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut
4,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut


In [5]:
# Load class mapping
with open(mapping_path, "r") as f:
    index_to_class = json.load(f)

# Convert keys back to integers
index_to_class = {
    int(k): v
    for k, v in index_to_class.items()
}

print("Class mapping loaded successfully.")

print(index_to_class)

Class mapping loaded successfully.
{0: 'BacterialLeafBlight', 1: 'BrownSpot', 2: 'Healthy', 3: 'Hispa', 4: 'LeafBlast', 5: 'LeafScald', 6: 'LeafSmut', 7: 'NarrowBrownLeafSpot', 8: 'NeckBlast'}


In [6]:
print("Validation class distribution:\n")

print(df_val["class_name"].value_counts())

Validation class distribution:

class_name
LeafSmut               400
BacterialLeafBlight    400
BrownSpot              400
LeafScald               91
NeckBlast               90
LeafBlast               89
NarrowBrownLeafSpot     88
Hispa                   86
Healthy                 86
Name: count, dtype: int64
